In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score
import json
from datetime import datetime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_ROOT  = "/content/drive/MyDrive/DeepLearning_Project/data/train_images"
DATA_ROOT2 = "/content/drive/MyDrive/DeepLearning_Project/data/train_masks"

AUTHENTIC_DIR = Path(DATA_ROOT) / "authentic"
FORGED_DIR    = Path(DATA_ROOT) / "forged"
MASK_DIR      = Path(DATA_ROOT2)

SPLIT_DIR = Path("/content/drive/MyDrive/DeepLearning_Project/splits")
TRAIN_CSV = SPLIT_DIR / "train_split.csv"
VAL_CSV   = SPLIT_DIR / "val_split.csv"

In [ ]:
# ============================================================================
# TRAINING PARAMETERS
# ============================================================================
TARGET_SIZE = (512, 512)  # (H,W)
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-4  # Lower LR for fine-tuning pre-trained model
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ImageNet normalization
MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
STD  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)


In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS (from our previous pipeline)
# ============================================================================
def decode_image(path):
    """Decode PNG/JPG robustly -> float32 RGB in [0,1]."""
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

def resize_image_mask(img, mask, target_size=TARGET_SIZE):
    h, w = target_size
    img  = tf.image.resize(img, (h, w), method="bilinear")
    mask = tf.image.resize(mask, (h, w), method="nearest")
    mask = tf.cast(mask > 0.5, tf.float32)
    return img, mask

def normalize_imagenet(img):
    return (img - MEAN) / STD

def mask_path_for_image_py(image_path_str):
    """Match image stem -> mask file in MASK_DIR."""
    p = Path(image_path_str)
    stem = p.stem
    direct = MASK_DIR / f"{stem}.npy"
    if direct.exists():
        return str(direct)
    candidates = sorted(MASK_DIR.glob(f"{stem}*.npy"))
    return str(candidates[0]) if len(candidates) else ""

def load_mask_npy_py(mask_path_str, orig_h, orig_w):
    """Load .npy instance mask (N,H,W), collapse -> (H,W) float32."""
    if (mask_path_str is None) or (mask_path_str == ""):
        return np.zeros((orig_h, orig_w), dtype=np.float32)

    inst = np.load(mask_path_str)
    if inst.ndim != 3:
        raise ValueError(f"Unexpected mask shape {inst.shape} for {mask_path_str}")

    bin_mask = (inst.max(axis=0) > 0).astype(np.float32)
    return bin_mask

def build_dataset(csv_path, training=True, data_fraction=1.0):
    """
    Build dataset with optional data fraction for ablation study.

    Args:
        csv_path: Path to CSV with image paths and labels
        training: Whether this is training set (enables shuffling)
        data_fraction: Fraction of data to use (0.25, 0.5, 0.75, 1.0)
    """
    df = pd.read_csv(csv_path)
    assert "path" in df.columns and "label" in df.columns

    # Sample data if needed (for training set size ablation)
    if data_fraction < 1.0 and training:
        df = df.sample(frac=data_fraction, random_state=SEED).reset_index(drop=True)
        print(f"Using {len(df)} samples ({data_fraction*100}% of training data)")

    paths = df["path"].astype(str).values
    labels = df["label"].astype(str).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=min(len(df), 2000), seed=SEED, reshuffle_each_iteration=True)

    def _load_example(path, label_str):
        img = decode_image(path)
        orig_shape = tf.shape(img)
        orig_h = orig_shape[0]
        orig_w = orig_shape[1]

        is_forged = tf.equal(label_str, tf.constant("forged"))

        def _load_forged_mask():
            mpath = tf.py_function(
                func=lambda x: mask_path_for_image_py(x.numpy().decode("utf-8")),
                inp=[path],
                Tout=tf.string
            )
            mpath.set_shape([])

            m = tf.py_function(
                func=lambda mp, h, w: load_mask_npy_py(mp.numpy().decode("utf-8"), int(h.numpy()), int(w.numpy())),
                inp=[mpath, orig_h, orig_w],
                Tout=tf.float32
            )
            m.set_shape([None, None])
            m = tf.expand_dims(m, axis=-1)
            return m

        def _load_auth_mask():
            m = tf.zeros([orig_h, orig_w, 1], dtype=tf.float32)
            return m

        mask = tf.cond(is_forged, _load_forged_mask, _load_auth_mask)
        img, mask = resize_image_mask(img, mask, TARGET_SIZE)
        img = normalize_imagenet(img)

        return img, mask

    ds = ds.map(_load_example, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
def build_deeplabv3plus(input_shape=(512, 512, 3), backbone='resnet50', weights='imagenet'):
    """
    Build DeepLabV3+ model with specified backbone.

    Args:
        input_shape: Input image shape
        backbone: 'resnet50' or 'resnet101'
        weights: 'imagenet' for pre-trained, None for from scratch
    """
    # Load backbone with or without pre-trained weights
    if backbone == 'resnet50':
        base_model = tf.keras.applications.ResNet50(
            weights=weights,
            include_top=False,
            input_shape=input_shape
        )
    elif backbone == 'resnet101':
        base_model = tf.keras.applications.ResNet101(
            weights=weights,
            include_top=False,
            input_shape=input_shape
        )
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")

    # Extract specific layers for skip connections
    # ResNet layers: conv1 -> conv2_x -> conv3_x -> conv4_x -> conv5_x

    # Low-level features (early layer for decoder)
    low_level_features = base_model.get_layer('conv2_block3_out').output  # 1/4 resolution

    # High-level features (final layer for ASPP)
    high_level_features = base_model.output  # 1/32 or 1/16 resolution

    # ============ ASPP Module (Atrous Spatial Pyramid Pooling) ============
    # Multiple parallel atrous convolutions with different rates
    x = high_level_features

    # 1x1 convolution
    conv_1x1 = tf.keras.layers.Conv2D(256, 1, padding='same', activation='relu')(x)

    # 3x3 convolutions with different dilation rates
    conv_3x3_rate6 = tf.keras.layers.Conv2D(256, 3, padding='same', dilation_rate=6, activation='relu')(x)
    conv_3x3_rate12 = tf.keras.layers.Conv2D(256, 3, padding='same', dilation_rate=12, activation='relu')(x)
    conv_3x3_rate18 = tf.keras.layers.Conv2D(256, 3, padding='same', dilation_rate=18, activation='relu')(x)

    # Global average pooling branch
    gap = tf.keras.layers.GlobalAveragePooling2D()(x)
    gap = tf.keras.layers.Reshape((1, 1, -1))(gap)
    gap = tf.keras.layers.Conv2D(256, 1, activation='relu')(gap)
    gap = tf.keras.layers.UpSampling2D(size=(x.shape[1], x.shape[2]), interpolation='bilinear')(gap)

    # Concatenate all ASPP branches
    aspp_out = tf.keras.layers.Concatenate()([conv_1x1, conv_3x3_rate6, conv_3x3_rate12, conv_3x3_rate18, gap])
    aspp_out = tf.keras.layers.Conv2D(256, 1, padding='same', activation='relu')(aspp_out)
    aspp_out = tf.keras.layers.Dropout(0.5)(aspp_out)

    # ============ Decoder ============
    # Upsample encoder output
    decoder = tf.keras.layers.UpSampling2D(size=(8, 8), interpolation='bilinear')(aspp_out)

    # Process low-level features
    low_level = tf.keras.layers.Conv2D(48, 1, padding='same', activation='relu')(low_level_features)

    # Concatenate with low-level features
    decoder = tf.keras.layers.Concatenate()([decoder, low_level])

    # Refine with 3x3 convolutions
    decoder = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(decoder)
    decoder = tf.keras.layers.Dropout(0.5)(decoder)
    decoder = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu')(decoder)
    decoder = tf.keras.layers.Dropout(0.1)(decoder)

    # Final upsampling to original resolution
    decoder = tf.keras.layers.UpSampling2D(size=(4, 4), interpolation='bilinear')(decoder)

    # Output layer
    output = tf.keras.layers.Conv2D(1, 1, activation='sigmoid', name='output')(decoder)

    model = tf.keras.Model(inputs=base_model.input, outputs=output)

    return model

In [ ]:
 #LOSS FUNCTIONS

def dice_loss(y_true, y_pred, smooth=1e-6):
    """Dice loss for binary segmentation."""
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dice = (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)
    return 1 - dice

def combined_loss(y_true, y_pred):
    """Combination of Dice loss and Binary Cross-Entropy."""
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

In [ ]:
# ============================================================================
# METRICS
# ============================================================================
class DiceCoefficient(tf.keras.metrics.Metric):
    """Dice coefficient metric."""
    def __init__(self, name='dice_coef', **kwargs):
        super().__init__(name=name, **kwargs)
        self.dice_sum = self.add_weight(name='dice_sum', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_f = tf.reshape(y_true, [-1])
        y_pred_f = tf.reshape(y_pred, [-1])
        intersection = tf.reduce_sum(y_true_f * y_pred_f)
        dice = (2. * intersection + 1e-6) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1e-6)
        self.dice_sum.assign_add(dice)
        self.count.assign_add(1.0)

    def result(self):
        return self.dice_sum / self.count

    def reset_state(self):
        self.dice_sum.assign(0.0)
        self.count.assign(0.0)

class IoU(tf.keras.metrics.Metric):
    """Intersection over Union metric."""
    def __init__(self, name='iou', **kwargs):
        super().__init__(name=name, **kwargs)
        self.iou_sum = self.add_weight(name='iou_sum', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_f = tf.reshape(y_true, [-1])
        y_pred_f = tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1])
        intersection = tf.reduce_sum(y_true_f * y_pred_f)
        union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
        iou = (intersection + 1e-6) / (union + 1e-6)
        self.iou_sum.assign_add(iou)
        self.count.assign_add(1.0)

    def result(self):
        return self.iou_sum / self.count

    def reset_state(self):
        self.iou_sum.assign(0.0)
        self.count.assign(0.0)

In [ ]:
 #THRESHOLD ANALYSIS FUNCTION

def analyze_thresholds(model, val_ds, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]):
    """
    Analyze performance across different binarization thresholds.

    Returns:
        dict: Dictionary with threshold -> metrics mapping
    """
    print("\n" + "="*60)
    print("THRESHOLD ANALYSIS")
    print("="*60)

    results = {}

    # Collect all predictions and ground truth
    all_preds = []
    all_masks = []

    for images, masks in val_ds:
        preds = model.predict(images, verbose=0)
        all_preds.append(preds)
        all_masks.append(masks.numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)

    # Test each threshold
    for thresh in thresholds:
        binary_preds = (all_preds > thresh).astype(np.float32)

        # Calculate metrics
        dice_scores = []
        iou_scores = []

        for i in range(len(all_masks)):
            pred_flat = binary_preds[i].flatten()
            mask_flat = all_masks[i].flatten()

            intersection = np.sum(pred_flat * mask_flat)
            dice = (2. * intersection + 1e-6) / (np.sum(pred_flat) + np.sum(mask_flat) + 1e-6)

            union = np.sum(pred_flat) + np.sum(mask_flat) - intersection
            iou = (intersection + 1e-6) / (union + 1e-6)

            dice_scores.append(dice)
            iou_scores.append(iou)

        avg_dice = np.mean(dice_scores)
        avg_iou = np.mean(iou_scores)

        results[thresh] = {
            'dice': avg_dice,
            'iou': avg_iou
        }

        print(f"Threshold: {thresh:.2f} | Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f}")

    return results


In [ ]:

def train_model(model_name, backbone, weights, data_fraction=1.0, save_dir="/content/drive/MyDrive/DeepLearning_Project/models"):
    """
    Train a DeepLabV3+ model with specified configuration.

    Args:
        model_name: Name for saving the model
        backbone: 'resnet50' or 'resnet101'
        weights: 'imagenet' or None
        data_fraction: Fraction of training data to use
        save_dir: Directory to save models and results
    """
    print("\n" + "="*60)
    print(f"Training: {model_name}")
    print(f"Backbone: {backbone}, Weights: {weights}, Data: {data_fraction*100}%")
    print("="*60 + "\n")

    # Create save directory
    save_path = Path(save_dir) / model_name
    save_path.mkdir(parents=True, exist_ok=True)

    # Build datasets
    train_ds = build_dataset(TRAIN_CSV, training=True, data_fraction=data_fraction)
    val_ds = build_dataset(VAL_CSV, training=False)

    # Build model
    model = build_deeplabv3plus(
        input_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 3),
        backbone=backbone,
        weights=weights
    )

    # Compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=combined_loss,
        metrics=[DiceCoefficient(), IoU()]
    )

    # Callbacks
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(save_path / 'best_model.h5'),
            monitor='val_dice_coef',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_dice_coef',
            mode='max',
            patience=10,
            verbose=1,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            verbose=1,
            min_lr=1e-7
        ),
        tf.keras.callbacks.CSVLogger(str(save_path / 'training_log.csv'))
    ]

    # Train
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # Save training history
    with open(save_path / 'history.json', 'w') as f:
        json.dump({k: [float(v) for v in values] for k, values in history.history.items()}, f)

    # Plot training curves
    plot_training_curves(history, save_path)

    return model, history



def plot_training_curves(history, save_path):
    """Plot and save training curves."""
    fig, axes = plt.subplots(2,2, figsize=(15, 10))

    # Loss
    axes[0, 0].plot(history.history['loss'], label='Train Loss')
    axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Dice
    axes[0, 1].plot(history.history['dice_coef'], label='Train Dice')
    axes[0, 1].plot(history.history['val_dice_coef'], label='Val Dice')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # IoU
    axes[1, 0].plot(history.history['iou'], label='Train IoU')
    axes[1, 0].plot(history.history['val_iou'], label='Val IoU')
    axes[1, 0].set_title('IoU')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('IoU')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # Learning rate (if available)
    #if 'lr' in history.history:
    #    axes[1, 1].plot(history.history['lr'])
    #    axes[1, 1].set_title('Learning Rate')
    #    axes[1, 1].set_xlabel('Epoch')
    #    axes[1, 1].set_ylabel('LR')
    #    axes[1, 1].set_yscale('log')
    #    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(save_path / 'training_curves.png', dpi=150)
    plt.close()

In [ ]:
def ablation_backbone():
    """Compare ResNet50 vs ResNet101 backbones."""
    print("\n" + "="*70)
    print("ABLATION STUDY 1: BACKBONE COMPARISON")
    print("="*70)

    results = {}

    # ResNet50
    #model_r50, history_r50 = train_model(
    #    model_name="deeplabv3plus_resnet50",
    #    backbone='resnet50',
    #    weights='imagenet',
    #    data_fraction=1.0
    #)
    results['resnet50'] = {
        'val_dice': max(history.history['val_dice_coef']),
        'val_iou': max(history.history['val_iou']),
        'val_loss': min(history.history['val_loss'])
    }

    # ResNet101
    model_r101, history_r101 = train_model(
        model_name="deeplabv3plus_resnet101",
        backbone='resnet101',
        weights='imagenet',
        data_fraction=1.0
    )
    results['resnet101'] = {
        'val_dice': max(history_r101.history['val_dice_coef']),
        'val_iou': max(history_r101.history['val_iou']),
        'val_loss': min(history_r101.history['val_loss'])
    }

    # Save comparison
    with open('/content/drive/MyDrive/DeepLearning_Project/models/ablation_backbone_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    print("\n" + "="*70)
    print("BACKBONE COMPARISON RESULTS:")
    print("="*70)
    for backbone, metrics in results.items():
        print(f"\n{backbone.upper()}:")
        print(f"  Best Val Dice: {metrics['val_dice']:.4f}")
        print(f"  Best Val IoU:  {metrics['val_iou']:.4f}")
        print(f"  Best Val Loss: {metrics['val_loss']:.4f}")

    return results

In [ ]:
def ablation_threshold():
    """Analyze performance across different binarization thresholds."""
    print("\n" + "="*70)
    print("ABLATION STUDY 2: THRESHOLD ANALYSIS")
    print("="*70)

    # Load best model from backbone comparison (use ResNet50 for speed)
    model = tf.keras.models.load_model(
        '/content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet50_full_data/best_model.h5',
        custom_objects={
            'combined_loss': combined_loss,
            'DiceCoefficient': DiceCoefficient,
            'IoU': IoU
        }
    )

    val_ds = build_dataset(VAL_CSV, training=False)

    # Analyze thresholds
    thresholds = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
    results = analyze_thresholds(model, val_ds, thresholds)

    # Convert float32 values to standard Python floats for JSON serialization
    serializable_results = {}
    for thresh, metrics in results.items():
        serializable_results[str(thresh)] = {
            'dice': float(metrics['dice']),
            'iou': float(metrics['iou'])
        }

    # Save results
    with open('/content/drive/MyDrive/DeepLearning_Project/models/ablation_threshold_results.json', 'w') as f:
        json.dump(serializable_results, f, indent=2)

    # Plot threshold analysis
    threshs = list(results.keys())
    dice_scores = [results[t]['dice'] for t in threshs]
    iou_scores = [results[t]['iou'] for t in threshs]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(threshs, dice_scores, 'o-', linewidth=2, markersize=8)
    plt.xlabel('Threshold', fontsize=12)
    plt.ylabel('Dice Score', fontsize=12)
    plt.title('Dice Score vs Threshold', fontsize=14)
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(threshs, iou_scores, 'o-', linewidth=2, color='orange', markersize=8)
    plt.xlabel('Threshold', fontsize=12)
    plt.ylabel('IoU Score', fontsize=12)
    plt.title('IoU vs Threshold', fontsize=14)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/DeepLearning_Project/models/threshold_analysis.png', dpi=150)
    plt.close()

    # Find optimal threshold
    best_thresh_dice = max(results.keys(), key=lambda t: results[t]['dice'])
    best_thresh_iou = max(results.keys(), key=lambda t: results[t]['iou'])

    print("\n" + "="*70)
    print("OPTIMAL THRESHOLDS:")
    print("="*70)
    print(f"Best for Dice: {best_thresh_dice:.2f} (Dice: {results[best_thresh_dice]['dice']:.4f})")
    print(f"Best for IoU:  {best_thresh_iou:.2f} (IoU: {results[best_thresh_iou]['iou']:.4f})")

    return results

In [ ]:
def ablation_training_size():
    """Compare performance with different training set sizes."""
    print("\n" + "="*70)
    print("ABLATION STUDY 3: TRAINING SET SIZE")
    print("="*70)

    results = {}
    fractions = [0.25, 0.5, 0.75, 1.0]

    for frac in fractions:
        model, history = train_model(
            model_name=f"deeplabv3plus_data{int(frac*100)}",
            backbone='resnet50',  # Use ResNet50 for consistency
            weights='imagenet',
            data_fraction=frac
        )

        results[f"{int(frac*100)}%"] = {
            'fraction': frac,
            'val_dice': max(history.history['val_dice_coef']),
            'val_iou': max(history.history['val_iou']),
            'val_loss': min(history.history['val_loss'])
        }

    # Save results
    with open('/content/drive/MyDrive/DeepLearning_Project/models/ablation_training_size_results.json', 'w') as f:
        json.dump(results, f, indent=2)

    # Plot data efficiency
    labels = list(results.keys())
    dice_scores = [results[l]['val_dice'] for l in labels]
    iou_scores = [results[l]['val_iou'] for l in labels]

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(labels, dice_scores, 'o-', linewidth=2, markersize=8)
    plt.xlabel('Training Data Size', fontsize=12)
    plt.ylabel('Val Dice Score', fontsize=12)
    plt.title('Data Efficiency: Dice Score', fontsize=14)
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(labels, iou_scores, 'o-', linewidth=2, color='orange', markersize=8)
    plt.xlabel('Training Data Size', fontsize=12)
    plt.ylabel('Val IoU Score', fontsize=12)
    plt.title('Data Efficiency: IoU Score', fontsize=14)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/DeepLearning_Project/models/training_size_analysis.png', dpi=150)
    plt.close()

    print("\n" + "="*70)
    print("TRAINING SET SIZE RESULTS:")
    print("="*70)
    for size, metrics in results.items():
        print(f"\n{size} of data:")
        print(f"  Val Dice: {metrics['val_dice']:.4f}")
        print(f"  Val IoU:  {metrics['val_iou']:.4f}")

    return results


In [ ]:
model, history = train_model(
    model_name="deeplabv3plus_resnet50_full_data",
    backbone='resnet50',
    weights='imagenet',
    data_fraction=1.0
)


Training: deeplabv3plus_resnet50_full_data
Backbone: resnet50, Weights: imagenet, Data: 100.0%

Epoch 1/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - dice_coef: 0.1772 - iou: 0.1126 - loss: 0.9716
Epoch 1: val_dice_coef improved from -inf to 0.19789, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet50_full_data/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 363s 613ms/step - dice_coef: 0.1773 - iou: 0.1127 - loss: 0.9714 - val_dice_coef: 0.1979 - val_iou: 0.1266 - val_loss: 0.9597 - learning_rate: 1.0000e-04
Epoch 2/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coef: 0.3743 - iou: 0.2691 - loss: 0.7449
Epoch 2: val_dice_coef improved from 0.19789 to 0.31370, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet50_full_data/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 108s 237ms/step - dice_coef: 0.3743 - iou: 0.2691 - loss: 0.7449 - val_dice_coef: 0.3137 - val_iou: 0.2288 - val_loss: 0.7941 - learning_rate: 1.0000e-04
Epoch 3/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step - dice_coef: 0.4197 - iou: 0.3047 - loss: 0.6810
Epoch 3: val_dice_coef improved from 0.31370 to 0.35690, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet50_full_data/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 108s 240ms/step - dice_coef: 0.4198 - iou: 0.3048 - loss: 0.6810 - val_dice_coef: 0.3569 - val_iou: 0.2504 - val_loss: 0.7590 - learning_rate: 1.0000e-04
Epoch 4/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - dice_coef: 0.4492 - iou: 0.3292 - loss: 0.6544
Epoch 4: val_dice_coef improved from 0.35690 to 0.38373, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet50_full_data/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 108s 236ms/step - dice_coef: 0.4492 - iou: 0.3292 - loss: 0.6544 - val_dice_coef: 0.3837 - val_iou: 0.2789 - val_loss: 0.7299 - learning_rate: 1.0000e-04
Epoch 5/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - dice_coef: 0.4981 - iou: 0.3741 - loss: 0.5957
Epoch 5: val_dice_coef did not improve from 0.38373
449/449 ━━━━━━━━━━━━━━━━━━━━ 103s 225ms/step - dice_coef: 0.4981 - iou: 0.3741 - loss: 0.5957 - val_dice_coef: 0.3552 - val_iou: 0.2475 - val_loss: 0.7595 - learning_rate: 1.0000e-04
Epoch 6/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.5057 - iou: 0.3820 - loss: 0.5824
Epoch 6: val_dice_coef did not improve from 0.38373
449/449 ━━━━━━━━━━━━━━━━━━━━ 93s 207ms/step - dice_coef: 0.5057 - iou: 0.3821 - loss: 0.5823 - val_dice_coef: 0.3810 - val_iou: 0.2744 - val_loss: 0.7212 - learning_rate: 1.0000e-04
Epoch 7/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.5490 - iou: 0.4239 - loss: 0.5388
Epoch 7: val_dice_coef improved from 0.

...

449/449 ━━━━━━━━━━━━━━━━━━━━ 108s 240ms/step - dice_coef: 0.5701 - iou: 0.4422 - loss: 0.5109 - val_dice_coef: 0.4138 - val_iou: 0.3020 - val_loss: 0.6985 - learning_rate: 1.0000e-04
Epoch 9/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step - dice_coef: 0.5857 - iou: 0.4602 - loss: 0.4922
Epoch 9: val_dice_coef improved from 0.41382 to 0.41903, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet50_full_data/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 108s 240ms/step - dice_coef: 0.5858 - iou: 0.4603 - loss: 0.4922 - val_dice_coef: 0.4190 - val_iou: 0.2946 - val_loss: 0.7142 - learning_rate: 1.0000e-04
Epoch 10/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - dice_coef: 0.6395 - iou: 0.5134 - loss: 0.4317
Epoch 10: val_dice_coef did not improve from 0.41903
449/449 ━━━━━━━━━━━━━━━━━━━━ 104s 227ms/step - dice_coef: 0.6395 - iou: 0.5134 - loss: 0.4317 - val_dice_coef: 0.3770 - val_iou: 0.2688 - val_loss: 0.7420 - learning_rate: 1.0000e-04
Epoch 11/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.6410 - iou: 0.5223 - loss: 0.4281
Epoch 11: val_dice_coef did not improve from 0.41903
449/449 ━━━━━━━━━━━━━━━━━━━━ 93s 206ms/step - dice_coef: 0.6410 - iou: 0.5223 - loss: 0.4282 - val_dice_coef: 0.4030 - val_iou: 0.2957 - val_loss: 0.6955 - learning_rate: 1.0000e-04
Epoch 12/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.6481 - iou: 0.5310 - loss: 0.4181
Epoch 12: val_dice_coef did not im

449/449 ━━━━━━━━━━━━━━━━━━━━ 99s 219ms/step - dice_coef: 0.6933 - iou: 0.5782 - loss: 0.3672 - val_dice_coef: 0.4367 - val_iou: 0.3140 - val_loss: 0.6819 - learning_rate: 1.0000e-04
Epoch 15/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step - dice_coef: 0.6617 - iou: 0.5524 - loss: 0.4004
Epoch 15: val_dice_coef did not improve from 0.43668
449/449 ━━━━━━━━━━━━━━━━━━━━ 102s 228ms/step - dice_coef: 0.6618 - iou: 0.5524 - loss: 0.4004 - val_dice_coef: 0.3966 - val_iou: 0.2739 - val_loss: 0.7438 - learning_rate: 1.0000e-04
Epoch 16/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step - dice_coef: 0.6968 - iou: 0.5800 - loss: 0.3677
Epoch 16: val_dice_coef did not improve from 0.43668
449/449 ━━━━━━━━━━━━━━━━━━━━ 93s 207ms/step - dice_coef: 0.6968 - iou: 0.5800 - loss: 0.3677 - val_dice_coef: 0.4321 - val_iou: 0.3050 - val_loss: 0.7226 - learning_rate: 1.0000e-04
Epoch 17/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.7068 - iou: 0.5964 - loss: 0.3487
Epoch 17: val_dice_coef improved fr

449/449 ━━━━━━━━━━━━━━━━━━━━ 98s 218ms/step - dice_coef: 0.7068 - iou: 0.5965 - loss: 0.3487 - val_dice_coef: 0.4445 - val_iou: 0.3209 - val_loss: 0.6796 - learning_rate: 1.0000e-04
Epoch 18/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - dice_coef: 0.7496 - iou: 0.6449 - loss: 0.2972
Epoch 18: val_dice_coef did not improve from 0.44448
449/449 ━━━━━━━━━━━━━━━━━━━━ 103s 230ms/step - dice_coef: 0.7496 - iou: 0.6449 - loss: 0.2972 - val_dice_coef: 0.4390 - val_iou: 0.3134 - val_loss: 0.6984 - learning_rate: 1.0000e-04
Epoch 19/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.7372 - iou: 0.6314 - loss: 0.3121
Epoch 19: val_dice_coef did not improve from 0.44448
449/449 ━━━━━━━━━━━━━━━━━━━━ 93s 207ms/step - dice_coef: 0.7372 - iou: 0.6314 - loss: 0.3121 - val_dice_coef: 0.4179 - val_iou: 0.2898 - val_loss: 0.7345 - learning_rate: 1.0000e-04
Epoch 20/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - dice_coef: 0.7445 - iou: 0.6405 - loss: 0.3065
Epoch 20: val_dice_coef improved fr

449/449 ━━━━━━━━━━━━━━━━━━━━ 99s 220ms/step - dice_coef: 0.7444 - iou: 0.6405 - loss: 0.3065 - val_dice_coef: 0.4458 - val_iou: 0.3192 - val_loss: 0.6931 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 20.


In [ ]:
ablation_backbone()


ABLATION STUDY 1: BACKBONE COMPARISON

Training: deeplabv3plus_resnet101
Backbone: resnet101, Weights: imagenet, Data: 100.0%

171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - dice_coef: 0.1678 - iou: 0.1123 - loss: 0.9960
Epoch 1: val_dice_coef improved from -inf to 0.16683, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 281s 374ms/step - dice_coef: 0.1679 - iou: 0.1124 - loss: 0.9958 - val_dice_coef: 0.1668 - val_iou: 0.1095 - val_loss: 0.9938 - learning_rate: 1.0000e-04
Epoch 2/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step - dice_coef: 0.3485 - iou: 0.2493 - loss: 0.7766
Epoch 2: val_dice_coef improved from 0.16683 to 0.27729, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 120s 266ms/step - dice_coef: 0.3485 - iou: 0.2493 - loss: 0.7766 - val_dice_coef: 0.2773 - val_iou: 0.2105 - val_loss: 0.8466 - learning_rate: 1.0000e-04
Epoch 3/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step - dice_coef: 0.3974 - iou: 0.2869 - loss: 0.7058
Epoch 3: val_dice_coef improved from 0.27729 to 0.32346, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 138s 302ms/step - dice_coef: 0.3975 - iou: 0.2869 - loss: 0.7057 - val_dice_coef: 0.3235 - val_iou: 0.2395 - val_loss: 0.7888 - learning_rate: 1.0000e-04
Epoch 4/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - dice_coef: 0.4328 - iou: 0.3166 - loss: 0.6714
Epoch 4: val_dice_coef did not improve from 0.32346
449/449 ━━━━━━━━━━━━━━━━━━━━ 128s 278ms/step - dice_coef: 0.4328 - iou: 0.3166 - loss: 0.6714 - val_dice_coef: 0.3019 - val_iou: 0.2208 - val_loss: 0.8328 - learning_rate: 1.0000e-04
Epoch 5/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step - dice_coef: 0.4433 - iou: 0.3261 - loss: 0.6584
Epoch 5: val_dice_coef improved from 0.32346 to 0.38849, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 118s 263ms/step - dice_coef: 0.4433 - iou: 0.3261 - loss: 0.6584 - val_dice_coef: 0.3885 - val_iou: 0.2779 - val_loss: 0.7210 - learning_rate: 1.0000e-04
Epoch 6/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step - dice_coef: 0.4647 - iou: 0.3443 - loss: 0.6291
Epoch 6: val_dice_coef did not improve from 0.38849
449/449 ━━━━━━━━━━━━━━━━━━━━ 128s 285ms/step - dice_coef: 0.4648 - iou: 0.3443 - loss: 0.6291 - val_dice_coef: 0.3308 - val_iou: 0.2329 - val_loss: 0.7968 - learning_rate: 1.0000e-04
Epoch 7/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step - dice_coef: 0.5090 - iou: 0.3856 - loss: 0.5844
Epoch 7: val_dice_coef improved from 0.38849 to 0.40504, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


...

449/449 ━━━━━━━━━━━━━━━━━━━━ 119s 264ms/step - dice_coef: 0.5926 - iou: 0.4782 - loss: 0.4806 - val_dice_coef: 0.4284 - val_iou: 0.3164 - val_loss: 0.6818 - learning_rate: 1.0000e-04
Epoch 14/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - dice_coef: 0.6550 - iou: 0.5355 - loss: 0.4130
Epoch 14: val_dice_coef improved from 0.42842 to 0.43539, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 136s 303ms/step - dice_coef: 0.6550 - iou: 0.5354 - loss: 0.4131 - val_dice_coef: 0.4354 - val_iou: 0.3085 - val_loss: 0.6843 - learning_rate: 1.0000e-04
Epoch 15/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - dice_coef: 0.6555 - iou: 0.5436 - loss: 0.4081
Epoch 15: val_dice_coef did not improve from 0.43539
449/449 ━━━━━━━━━━━━━━━━━━━━ 127s 282ms/step - dice_coef: 0.6555 - iou: 0.5436 - loss: 0.4081 - val_dice_coef: 0.4038 - val_iou: 0.2782 - val_loss: 0.7463 - learning_rate: 1.0000e-04
Epoch 16/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - dice_coef: 0.7058 - iou: 0.5930 - loss: 0.3597
Epoch 16: val_dice_coef did not improve from 0.43539
449/449 ━━━━━━━━━━━━━━━━━━━━ 110s 246ms/step - dice_coef: 0.7058 - iou: 0.5930 - loss: 0.3597 - val_dice_coef: 0.4147 - val_iou: 0.2931 - val_loss: 0.7297 - learning_rate: 1.0000e-04
Epoch 17/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - dice_coef: 0.6911 - iou: 0.5801 - loss: 0.3697
Epoch 17: val_dice_coef did not i

449/449 ━━━━━━━━━━━━━━━━━━━━ 120s 268ms/step - dice_coef: 0.7270 - iou: 0.6233 - loss: 0.3247 - val_dice_coef: 0.4569 - val_iou: 0.3298 - val_loss: 0.6660 - learning_rate: 5.0000e-05
Epoch 20/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - dice_coef: 0.7699 - iou: 0.6733 - loss: 0.2739
Epoch 20: val_dice_coef improved from 0.45687 to 0.46470, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_resnet101/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 137s 299ms/step - dice_coef: 0.7699 - iou: 0.6733 - loss: 0.2739 - val_dice_coef: 0.4647 - val_iou: 0.3313 - val_loss: 0.6649 - learning_rate: 5.0000e-05
Restoring model weights from the end of the best epoch: 20.

BACKBONE COMPARISON RESULTS:

RESNET50:
  Best Val Dice: 0.4458
  Best Val IoU:  0.3209
  Best Val Loss: 0.6796

RESNET101:
  Best Val Dice: 0.4647
  Best Val IoU:  0.3313
  Best Val Loss: 0.6649


{'resnet50': {'val_dice': 0.44578874111175537,
  'val_iou': 0.32087990641593933,
  'val_loss': 0.6796075701713562},
 'resnet101': {'val_dice': 0.4647018611431122,
  'val_iou': 0.3313458263874054,
  'val_loss': 0.6649486422538757}}

In [ ]:
ablation_threshold()


ABLATION STUDY 2: THRESHOLD ANALYSIS



THRESHOLD ANALYSIS
Threshold: 0.30 | Dice: 0.5614 | IoU: 0.5076
Threshold: 0.35 | Dice: 0.5620 | IoU: 0.5083
Threshold: 0.40 | Dice: 0.5635 | IoU: 0.5100
Threshold: 0.45 | Dice: 0.5640 | IoU: 0.5106
Threshold: 0.50 | Dice: 0.5664 | IoU: 0.5131
Threshold: 0.55 | Dice: 0.5658 | IoU: 0.5127
Threshold: 0.60 | Dice: 0.5671 | IoU: 0.5141
Threshold: 0.65 | Dice: 0.5664 | IoU: 0.5135
Threshold: 0.70 | Dice: 0.5664 | IoU: 0.5137

OPTIMAL THRESHOLDS:
Best for Dice: 0.60 (Dice: 0.5671)
Best for IoU:  0.60 (IoU: 0.5141)


{0.3: {'dice': np.float32(0.5613904), 'iou': np.float32(0.50764817)},
 0.35: {'dice': np.float32(0.56196696), 'iou': np.float32(0.5083439)},
 0.4: {'dice': np.float32(0.56348634), 'iou': np.float32(0.5099823)},
 0.45: {'dice': np.float32(0.5639915), 'iou': np.float32(0.5106023)},
 0.5: {'dice': np.float32(0.56636626), 'iou': np.float32(0.51310414)},
 0.55: {'dice': np.float32(0.56580484), 'iou': np.float32(0.51267004)},
 0.6: {'dice': np.float32(0.56709427), 'iou': np.float32(0.51408696)},
 0.65: {'dice': np.float32(0.56635463), 'iou': np.float32(0.51346976)},
 0.7: {'dice': np.float32(0.56643534), 'iou': np.float32(0.5136953)}}

In [ ]:
ablation_training_size()


ABLATION STUDY 3: TRAINING SET SIZE

Training: deeplabv3plus_data25
Backbone: resnet50, Weights: imagenet, Data: 25.0%

Using 897 samples (25.0% of training data)
Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step - dice_coef: 0.1082 - iou: 0.0516 - loss: 1.0382
Epoch 1: val_dice_coef improved from -inf to 0.00888, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data25/best_model.h5


113/113 ━━━━━━━━━━━━━━━━━━━━ 138s 647ms/step - dice_coef: 0.1088 - iou: 0.0521 - loss: 1.0375 - val_dice_coef: 0.0089 - val_iou: 3.0365e-11 - val_loss: 1.1572 - learning_rate: 1.0000e-04
Epoch 2/20
112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - dice_coef: 0.3178 - iou: 0.2252 - loss: 0.8057
Epoch 2: val_dice_coef improved from 0.00888 to 0.10220, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data25/best_model.h5


113/113 ━━━━━━━━━━━━━━━━━━━━ 38s 339ms/step - dice_coef: 0.3175 - iou: 0.2251 - loss: 0.8059 - val_dice_coef: 0.1022 - val_iou: 0.0648 - val_loss: 1.0395 - learning_rate: 1.0000e-04
Epoch 3/20
112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - dice_coef: 0.3854 - iou: 0.2813 - loss: 0.7204
Epoch 3: val_dice_coef improved from 0.10220 to 0.19175, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data25/best_model.h5


113/113 ━━━━━━━━━━━━━━━━━━━━ 47s 405ms/step - dice_coef: 0.3856 - iou: 0.2814 - loss: 0.7203 - val_dice_coef: 0.1917 - val_iou: 0.1235 - val_loss: 0.9590 - learning_rate: 1.0000e-04
Epoch 4/20
112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step - dice_coef: 0.4137 - iou: 0.3100 - loss: 0.6879
Epoch 4: val_dice_coef improved from 0.19175 to 0.23886, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data25/best_model.h5


113/113 ━━━━━━━━━━━━━━━━━━━━ 48s 420ms/step - dice_coef: 0.4144 - iou: 0.3106 - loss: 0.6873 - val_dice_coef: 0.2389 - val_iou: 0.1587 - val_loss: 0.9054 - learning_rate: 1.0000e-04
Epoch 5/20
112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - dice_coef: 0.4868 - iou: 0.3633 - loss: 0.6072
Epoch 5: val_dice_coef did not improve from 0.23886
113/113 ━━━━━━━━━━━━━━━━━━━━ 42s 354ms/step - dice_coef: 0.4868 - iou: 0.3634 - loss: 0.6071 - val_dice_coef: 0.2040 - val_iou: 0.1293 - val_loss: 0.9624 - learning_rate: 1.0000e-04
Epoch 6/20
112/113 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step - dice_coef: 0.5476 - iou: 0.4185 - loss: 0.5382
Epoch 6: val_dice_coef improved from 0.23886 to 0.23940, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data25/best_model.h5


...

449/449 ━━━━━━━━━━━━━━━━━━━━ 104s 231ms/step - dice_coef: 0.6848 - iou: 0.5705 - loss: 0.3772 - val_dice_coef: 0.4358 - val_iou: 0.3053 - val_loss: 0.7051 - learning_rate: 1.0000e-04
Epoch 15/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - dice_coef: 0.6909 - iou: 0.5789 - loss: 0.3657
Epoch 15: val_dice_coef did not improve from 0.43581

Epoch 15: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
449/449 ━━━━━━━━━━━━━━━━━━━━ 106s 237ms/step - dice_coef: 0.6909 - iou: 0.5790 - loss: 0.3657 - val_dice_coef: 0.4189 - val_iou: 0.2896 - val_loss: 0.7323 - learning_rate: 1.0000e-04
Epoch 16/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - dice_coef: 0.7579 - iou: 0.6522 - loss: 0.2930
Epoch 16: val_dice_coef improved from 0.43581 to 0.45045, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data100/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 103s 229ms/step - dice_coef: 0.7579 - iou: 0.6522 - loss: 0.2930 - val_dice_coef: 0.4504 - val_iou: 0.3173 - val_loss: 0.6879 - learning_rate: 5.0000e-05
Epoch 17/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - dice_coef: 0.7529 - iou: 0.6513 - loss: 0.2930
Epoch 17: val_dice_coef improved from 0.45045 to 0.45142, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data100/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 111s 247ms/step - dice_coef: 0.7530 - iou: 0.6513 - loss: 0.2929 - val_dice_coef: 0.4514 - val_iou: 0.3271 - val_loss: 0.6676 - learning_rate: 5.0000e-05
Epoch 18/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - dice_coef: 0.7745 - iou: 0.6785 - loss: 0.2658
Epoch 18: val_dice_coef improved from 0.45142 to 0.46448, saving model to /content/drive/MyDrive/DeepLearning_Project/models/deeplabv3plus_data100/best_model.h5


449/449 ━━━━━━━━━━━━━━━━━━━━ 112s 246ms/step - dice_coef: 0.7745 - iou: 0.6785 - loss: 0.2657 - val_dice_coef: 0.4645 - val_iou: 0.3320 - val_loss: 0.6756 - learning_rate: 5.0000e-05
Epoch 19/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - dice_coef: 0.7802 - iou: 0.6882 - loss: 0.2580
Epoch 19: val_dice_coef did not improve from 0.46448
449/449 ━━━━━━━━━━━━━━━━━━━━ 106s 233ms/step - dice_coef: 0.7802 - iou: 0.6882 - loss: 0.2580 - val_dice_coef: 0.4568 - val_iou: 0.3296 - val_loss: 0.6764 - learning_rate: 5.0000e-05
Epoch 20/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - dice_coef: 0.7914 - iou: 0.6970 - loss: 0.2483
Epoch 20: val_dice_coef did not improve from 0.46448
449/449 ━━━━━━━━━━━━━━━━━━━━ 98s 216ms/step - dice_coef: 0.7914 - iou: 0.6970 - loss: 0.2483 - val_dice_coef: 0.4496 - val_iou: 0.3206 - val_loss: 0.6888 - learning_rate: 5.0000e-05
Restoring model weights from the end of the best epoch: 18.

TRAINING SET SIZE RESULTS:

25% of data:
  Val Dice: 0.3403
  Val IoU:  0.228

{'25%': {'fraction': 0.25,
  'val_dice': 0.3403134346008301,
  'val_iou': 0.22819942235946655,
  'val_loss': 0.8066745400428772},
 '50%': {'fraction': 0.5,
  'val_dice': 0.40662887692451477,
  'val_iou': 0.28693920373916626,
  'val_loss': 0.741426944732666},
 '75%': {'fraction': 0.75,
  'val_dice': 0.438517302274704,
  'val_iou': 0.3098180294036865,
  'val_loss': 0.6943842172622681},
 '100%': {'fraction': 1.0,
  'val_dice': 0.4644799530506134,
  'val_iou': 0.3319602608680725,
  'val_loss': 0.6675873398780823}}